<a href="https://colab.research.google.com/github/i2mmmmm/train_project/blob/main/20230826_ar_c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LayerNormalization, MultiHeadAttention, Flatten
import tensorflow as tf
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Activation
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Flatten, Dense, Dropout
from tensorflow.keras.models import Model
from sklearn.preprocessing import StandardScaler

In [5]:
from google.colab import drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
s30_train = pd.read_csv('/content/drive/My Drive/철도/s30_train.csv')
s40_train = pd.read_csv('/content/drive/My Drive/철도/s40_train.csv')
s50_train = pd.read_csv('/content/drive/My Drive/철도/s50_train.csv')
s70_train = pd.read_csv('/content/drive/My Drive/철도/s70_train.csv')
s100_train = pd.read_csv('/content/drive/My Drive/철도/s100_train.csv')
c30_train = pd.read_csv('/content/drive/My Drive/철도/c30_train.csv')
c40_train = pd.read_csv('/content/drive/My Drive/철도/c40_train.csv')
c50_train = pd.read_csv('/content/drive/My Drive/철도/c50_train.csv')
c70_train = pd.read_csv('/content/drive/My Drive/철도/c70_train.csv')
c100_train = pd.read_csv('/content/drive/My Drive/철도/c100_train.csv')

In [7]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=['Curvature', 'Vertical offset', 'Cross level offset', 'A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W2_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W2_Y_A_axle_L', 'A__B1_W1_Z_A_axle_R'])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c30_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 27

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 26s 109ms/step - loss: 0.0016 - mae: 0.0243 - val_loss: 2.6741e-04 - val_mae: 0.0125
Epoch 2/50
157/157 [==============================] - 15s 96ms/step - loss: 6.4407e-04 - mae: 0.0144 - val_loss: 1.4533e-04 - val_mae: 0.0096
Epoch 3/50
157/157 [==============================] - 15s 98ms/step - loss: 4.7871e-04 - mae: 0.0123 - val_loss: 1.0081e-04 - val_mae: 0.0079
Epoch 4/50
157/157 [==============================] - 18s 115ms/step - loss: 4.1707e-04 - mae: 0.0112 - val_loss: 1.2103e-04 - val_mae: 0.0087
Epoch 5/50
157/157 [==============================] - 24s 151ms/step - loss: 3.7640e-04 - mae: 0.0107 - val_loss: 1.0700e-04 - val_mae: 0.0081
Epoch 6/50
157/157 [==============================] - 16s 104ms/step - loss: 3.9827e-04 - mae: 0.0101 - val_loss: 9.5238e-05 - val_mae: 0.0076
Epoch 7/50
157/157 [==============================] - 15s 96ms/step - loss: 3.5072e-04 - mae: 0.0095 - val_loss: 1.2275e-04 - val_mae: 0.0089
Epoch 

In [8]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=['A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W1_Z_A_axle_L', 'A__B1_W1_Z_A_axle_R', 'A__B1_W2_Z_A_axle_R', 'V_M1_B1_W1_L', 'V_M1_B1_W1_R', 'QL_M1_B1_W1', 'QR_M1_B1_W1'])
    y = data[["YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c30_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 26

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s2 = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 28s 108ms/step - loss: 0.0015 - mae: 0.0265 - val_loss: 5.6824e-04 - val_mae: 0.0192
Epoch 2/50
157/157 [==============================] - 15s 95ms/step - loss: 5.7714e-04 - mae: 0.0162 - val_loss: 4.1433e-04 - val_mae: 0.0164
Epoch 3/50
157/157 [==============================] - 18s 114ms/step - loss: 4.3851e-04 - mae: 0.0141 - val_loss: 4.6738e-04 - val_mae: 0.0173
Epoch 4/50
157/157 [==============================] - 18s 114ms/step - loss: 3.9270e-04 - mae: 0.0131 - val_loss: 3.0046e-04 - val_mae: 0.0140
Epoch 5/50
157/157 [==============================] - 16s 100ms/step - loss: 3.0495e-04 - mae: 0.0116 - val_loss: 2.5598e-04 - val_mae: 0.0128
Epoch 6/50
157/157 [==============================] - 30s 192ms/step - loss: 2.9616e-04 - mae: 0.0114 - val_loss: 2.8523e-04 - val_mae: 0.0136
Epoch 7/50
157/157 [==============================] - 16s 100ms/step - loss: 3.0360e-04 - mae: 0.0112 - val_loss: 3.0561e-04 - val_mae: 0.0140
Epoc

In [9]:
predictions_30c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_30c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [10]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=['Curvature', 'Vertical offset', 'Cross level offset', 'A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W2_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W2_Y_A_axle_L', 'A__B1_W1_Z_A_axle_R'])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c40_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 27

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 33s 150ms/step - loss: 8.4365e-04 - mae: 0.0196 - val_loss: 8.1955e-05 - val_mae: 0.0072
Epoch 2/50
157/157 [==============================] - 18s 113ms/step - loss: 1.9445e-04 - mae: 0.0100 - val_loss: 8.4174e-05 - val_mae: 0.0074
Epoch 3/50
157/157 [==============================] - 15s 98ms/step - loss: 1.4905e-04 - mae: 0.0085 - val_loss: 7.1478e-05 - val_mae: 0.0067
Epoch 4/50
157/157 [==============================] - 16s 102ms/step - loss: 1.2586e-04 - mae: 0.0076 - val_loss: 6.1405e-05 - val_mae: 0.0063
Epoch 5/50
157/157 [==============================] - 16s 101ms/step - loss: 1.0516e-04 - mae: 0.0071 - val_loss: 6.9869e-05 - val_mae: 0.0067
Epoch 6/50
157/157 [==============================] - 15s 98ms/step - loss: 9.3155e-05 - mae: 0.0067 - val_loss: 5.7612e-05 - val_mae: 0.0060
Epoch 7/50
157/157 [==============================] - 22s 140ms/step - loss: 8.6396e-05 - mae: 0.0064 - val_loss: 5.2791e-05 - val_mae: 0.0057
E

In [11]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=['A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W1_Z_A_axle_L', 'A__B1_W1_Z_A_axle_R', 'A__B1_W2_Z_A_axle_R', 'V_M1_B1_W1_L', 'V_M1_B1_W1_R', 'QL_M1_B1_W1', 'QR_M1_B1_W1'])
    y = data[["YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c40_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 26

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s2 = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 32s 146ms/step - loss: 0.0013 - mae: 0.0244 - val_loss: 3.4312e-04 - val_mae: 0.0147
Epoch 2/50
157/157 [==============================] - 18s 113ms/step - loss: 3.4173e-04 - mae: 0.0136 - val_loss: 2.4246e-04 - val_mae: 0.0122
Epoch 3/50
157/157 [==============================] - 16s 100ms/step - loss: 2.4604e-04 - mae: 0.0113 - val_loss: 1.9047e-04 - val_mae: 0.0108
Epoch 4/50
157/157 [==============================] - 18s 112ms/step - loss: 2.1190e-04 - mae: 0.0104 - val_loss: 1.8276e-04 - val_mae: 0.0108
Epoch 5/50
157/157 [==============================] - 16s 102ms/step - loss: 1.7648e-04 - mae: 0.0096 - val_loss: 1.5243e-04 - val_mae: 0.0097
Epoch 6/50
157/157 [==============================] - 16s 102ms/step - loss: 1.6160e-04 - mae: 0.0092 - val_loss: 1.2092e-04 - val_mae: 0.0087
Epoch 7/50
157/157 [==============================] - 22s 141ms/step - loss: 1.5773e-04 - mae: 0.0089 - val_loss: 1.1614e-04 - val_mae: 0.0084
Epo

In [12]:
predictions_40c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_40c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [13]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=['Curvature', 'Vertical offset', 'Cross level offset', 'A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W2_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W2_Y_A_axle_L', 'A__B1_W1_Z_A_axle_R'])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c50_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 27

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 28s 121ms/step - loss: 9.0843e-04 - mae: 0.0202 - val_loss: 8.2063e-05 - val_mae: 0.0072
Epoch 2/50
157/157 [==============================] - 16s 101ms/step - loss: 1.7350e-04 - mae: 0.0099 - val_loss: 8.7950e-05 - val_mae: 0.0075
Epoch 3/50
157/157 [==============================] - 16s 103ms/step - loss: 1.2229e-04 - mae: 0.0082 - val_loss: 7.7037e-05 - val_mae: 0.0069
Epoch 4/50
157/157 [==============================] - 16s 103ms/step - loss: 9.7989e-05 - mae: 0.0073 - val_loss: 7.1470e-05 - val_mae: 0.0066
Epoch 5/50
157/157 [==============================] - 18s 117ms/step - loss: 8.6039e-05 - mae: 0.0069 - val_loss: 5.2259e-05 - val_mae: 0.0057
Epoch 6/50
157/157 [==============================] - 20s 130ms/step - loss: 8.0779e-05 - mae: 0.0066 - val_loss: 4.9592e-05 - val_mae: 0.0055
Epoch 7/50
157/157 [==============================] - 19s 118ms/step - loss: 7.3711e-05 - mae: 0.0062 - val_loss: 4.7691e-05 - val_mae: 0.0054

In [14]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=['A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W1_Z_A_axle_L', 'A__B1_W1_Z_A_axle_R', 'A__B1_W2_Z_A_axle_R', 'V_M1_B1_W1_L', 'V_M1_B1_W1_R', 'QL_M1_B1_W1', 'QR_M1_B1_W1'])
    y = data[["YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c50_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 26

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s2 = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 28s 125ms/step - loss: 0.0012 - mae: 0.0240 - val_loss: 4.3040e-04 - val_mae: 0.0163
Epoch 2/50
157/157 [==============================] - 16s 101ms/step - loss: 2.9819e-04 - mae: 0.0130 - val_loss: 3.2823e-04 - val_mae: 0.0141
Epoch 3/50
157/157 [==============================] - 15s 96ms/step - loss: 2.2524e-04 - mae: 0.0111 - val_loss: 2.4540e-04 - val_mae: 0.0124
Epoch 4/50
157/157 [==============================] - 17s 111ms/step - loss: 1.8032e-04 - mae: 0.0098 - val_loss: 1.8491e-04 - val_mae: 0.0106
Epoch 5/50
157/157 [==============================] - 16s 100ms/step - loss: 1.6398e-04 - mae: 0.0094 - val_loss: 1.9183e-04 - val_mae: 0.0111
Epoch 6/50
157/157 [==============================] - 20s 127ms/step - loss: 1.4702e-04 - mae: 0.0089 - val_loss: 1.4801e-04 - val_mae: 0.0096
Epoch 7/50
157/157 [==============================] - 16s 100ms/step - loss: 1.3228e-04 - mae: 0.0084 - val_loss: 1.0856e-04 - val_mae: 0.0082
Epoc

In [15]:
predictions_50c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_50c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [16]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=['Curvature', 'Vertical offset', 'Cross level offset', 'A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W2_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W2_Y_A_axle_L', 'A__B1_W1_Z_A_axle_R'])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c70_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 27

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 33s 144ms/step - loss: 8.3017e-04 - mae: 0.0195 - val_loss: 7.5258e-05 - val_mae: 0.0069
Epoch 2/50
157/157 [==============================] - 16s 102ms/step - loss: 1.5607e-04 - mae: 0.0096 - val_loss: 7.4575e-05 - val_mae: 0.0068
Epoch 3/50
157/157 [==============================] - 16s 103ms/step - loss: 1.0919e-04 - mae: 0.0079 - val_loss: 7.1663e-05 - val_mae: 0.0067
Epoch 4/50
157/157 [==============================] - 16s 100ms/step - loss: 8.5490e-05 - mae: 0.0071 - val_loss: 5.8732e-05 - val_mae: 0.0060
Epoch 5/50
157/157 [==============================] - 19s 118ms/step - loss: 7.6431e-05 - mae: 0.0066 - val_loss: 7.2803e-05 - val_mae: 0.0067
Epoch 6/50
157/157 [==============================] - 16s 103ms/step - loss: 6.7718e-05 - mae: 0.0062 - val_loss: 4.9917e-05 - val_mae: 0.0055
Epoch 7/50
157/157 [==============================] - 20s 129ms/step - loss: 6.2326e-05 - mae: 0.0059 - val_loss: 5.0438e-05 - val_mae: 0.0055

In [17]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=['A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W1_Z_A_axle_L', 'A__B1_W1_Z_A_axle_R', 'A__B1_W2_Z_A_axle_R', 'V_M1_B1_W1_L', 'V_M1_B1_W1_R', 'QL_M1_B1_W1', 'QR_M1_B1_W1'])
    y = data[["YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c70_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 26

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s2 = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 27s 112ms/step - loss: 0.0012 - mae: 0.0241 - val_loss: 6.0138e-04 - val_mae: 0.0194
Epoch 2/50
157/157 [==============================] - 18s 117ms/step - loss: 2.9527e-04 - mae: 0.0130 - val_loss: 4.1632e-04 - val_mae: 0.0161
Epoch 3/50
157/157 [==============================] - 23s 145ms/step - loss: 2.1666e-04 - mae: 0.0110 - val_loss: 2.6189e-04 - val_mae: 0.0128
Epoch 4/50
157/157 [==============================] - 17s 106ms/step - loss: 1.8115e-04 - mae: 0.0099 - val_loss: 2.2980e-04 - val_mae: 0.0119
Epoch 5/50
157/157 [==============================] - 16s 104ms/step - loss: 1.5776e-04 - mae: 0.0092 - val_loss: 1.9451e-04 - val_mae: 0.0108
Epoch 6/50
157/157 [==============================] - 19s 119ms/step - loss: 1.4703e-04 - mae: 0.0088 - val_loss: 1.4757e-04 - val_mae: 0.0094
Epoch 7/50
157/157 [==============================] - 17s 106ms/step - loss: 1.2763e-04 - mae: 0.0083 - val_loss: 1.2826e-04 - val_mae: 0.0088
Epo

In [18]:
predictions_70c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_70c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [19]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=['Curvature', 'Vertical offset', 'Cross level offset', 'A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W2_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W2_Y_A_axle_L', 'A__B1_W1_Z_A_axle_R'])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c100_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 27

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 31s 139ms/step - loss: 8.6100e-04 - mae: 0.0198 - val_loss: 9.2019e-05 - val_mae: 0.0075
Epoch 2/50
157/157 [==============================] - 19s 123ms/step - loss: 1.5906e-04 - mae: 0.0096 - val_loss: 9.8101e-05 - val_mae: 0.0077
Epoch 3/50
157/157 [==============================] - 19s 120ms/step - loss: 1.0941e-04 - mae: 0.0080 - val_loss: 8.4501e-05 - val_mae: 0.0071
Epoch 4/50
157/157 [==============================] - 17s 107ms/step - loss: 8.8467e-05 - mae: 0.0072 - val_loss: 7.3119e-05 - val_mae: 0.0067
Epoch 5/50
157/157 [==============================] - 17s 109ms/step - loss: 7.8344e-05 - mae: 0.0067 - val_loss: 5.2336e-05 - val_mae: 0.0056
Epoch 6/50
157/157 [==============================] - 19s 123ms/step - loss: 6.9789e-05 - mae: 0.0063 - val_loss: 6.8388e-05 - val_mae: 0.0064
Epoch 7/50
157/157 [==============================] - 19s 121ms/step - loss: 6.2550e-05 - mae: 0.0060 - val_loss: 5.4433e-05 - val_mae: 0.0057

In [20]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=['A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W1_Z_A_axle_L', 'A__B1_W1_Z_A_axle_R', 'A__B1_W2_Z_A_axle_R', 'V_M1_B1_W1_L', 'V_M1_B1_W1_R', 'QL_M1_B1_W1', 'QR_M1_B1_W1'])
    y = data[["YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c100_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 26

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s2 = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 26s 108ms/step - loss: 0.0011 - mae: 0.0235 - val_loss: 3.7868e-04 - val_mae: 0.0154
Epoch 2/50
157/157 [==============================] - 22s 139ms/step - loss: 2.9717e-04 - mae: 0.0130 - val_loss: 3.2661e-04 - val_mae: 0.0143
Epoch 3/50
157/157 [==============================] - 16s 99ms/step - loss: 2.0910e-04 - mae: 0.0108 - val_loss: 1.7264e-04 - val_mae: 0.0103
Epoch 4/50
157/157 [==============================] - 17s 106ms/step - loss: 1.6458e-04 - mae: 0.0095 - val_loss: 1.9930e-04 - val_mae: 0.0111
Epoch 5/50
157/157 [==============================] - 17s 106ms/step - loss: 1.5300e-04 - mae: 0.0091 - val_loss: 1.6817e-04 - val_mae: 0.0101
Epoch 6/50
157/157 [==============================] - 16s 101ms/step - loss: 1.3830e-04 - mae: 0.0086 - val_loss: 1.1621e-04 - val_mae: 0.0085
Epoch 7/50
157/157 [==============================] - 18s 112ms/step - loss: 1.2700e-04 - mae: 0.0082 - val_loss: 1.1181e-04 - val_mae: 0.0083
Epoc

In [21]:
predictions_100c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_100c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [22]:
answer_sample = pd.read_csv('/content/drive/My Drive/철도/answer_sample.csv')
c30 = pd.concat([predictions_30c,predictions_30c2],axis=1)
c40 = pd.concat([predictions_40c,predictions_40c2],axis=1)
c50 = pd.concat([predictions_50c,predictions_50c2],axis=1)
c70 = pd.concat([predictions_70c,predictions_70c2],axis=1)
c100 = pd.concat([predictions_100c,predictions_100c2],axis=1)

In [23]:
c30  = pd.DataFrame(c30, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c40  = pd.DataFrame(c40, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c50  = pd.DataFrame(c50, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c70  = pd.DataFrame(c70, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c100  = pd.DataFrame(c100, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])

In [24]:
c30.columns = ['YL_M1_B1_W1_c30', 'YR_M1_B1_W1_c30', 'YL_M1_B1_W2_c30', 'YR_M1_B1_W2_c30']
c40.columns = ['YL_M1_B1_W1_c40', 'YR_M1_B1_W1_c40', 'YL_M1_B1_W2_c40', 'YR_M1_B1_W2_c40']
c50.columns = ['YL_M1_B1_W1_c50', 'YR_M1_B1_W1_c50', 'YL_M1_B1_W2_c50', 'YR_M1_B1_W2_c50']
c70.columns = ['YL_M1_B1_W1_c70', 'YR_M1_B1_W1_c70', 'YL_M1_B1_W2_c70', 'YR_M1_B1_W2_c70']
c100.columns = ['YL_M1_B1_W1_c100', 'YR_M1_B1_W1_c100', 'YL_M1_B1_W2_c100', 'YR_M1_B1_W2_c100']

In [25]:
answer2 = pd.concat([answer_sample.Distance,c30,c40,c50,c70,c100], axis=1)

In [26]:
answer2

,Distance,YL_M1_B1_W1_c30,YR_M1_B1_W1_c30,YL_M1_B1_W2_c30,YR_M1_B1_W2_c30,YL_M1_B1_W1_c40,YR_M1_B1_W1_c40,YL_M1_B1_W2_c40,YR_M1_B1_W2_c40,YL_M1_B1_W1_c50,...,YL_M1_B1_W2_c50,YR_M1_B1_W2_c50,YL_M1_B1_W1_c70,YR_M1_B1_W1_c70,YL_M1_B1_W2_c70,YR_M1_B1_W2_c70,YL_M1_B1_W1_c100,YR_M1_B1_W1_c100,YL_M1_B1_W2_c100,YR_M1_B1_W2_c100
0,2500.25,0.021011,0.000561,0.044664,-0.030090,0.002714,0.007235,-0.004830,0.008967,0.009605,...,-0.002854,0.007505,0.010698,0.005484,-0.004044,0.008491,0.009935,0.006552,-0.003808,0.008449
1,2500.50,-0.003516,0.011637,-0.006212,0.010346,-0.002044,0.008441,0.011871,-0.007901,-0.001520,...,0.009938,-0.005174,-0.006434,0.009810,0.010753,-0.005876,-0.004632,0.008645,0.008023,-0.002847
2,2500.75,-0.006366,0.011134,0.003692,0.001595,-0.007339,0.011898,0.006406,-0.003792,-0.007177,...,0.006340,-0.002528,-0.010609,0.010505,0.006946,-0.004182,-0.010056,0.010169,0.006999,-0.002435
3,2501.00,-0.004510,0.007682,0.001047,0.003195,-0.004953,0.008980,0.003915,-0.001714,-0.005643,...,0.005526,-0.002172,-0.007203,0.006657,0.005290,-0.003604,-0.004978,0.004826,0.004712,-0.000885
4,2501.25,-0.003268,0.005388,0.002483,0.001886,-0.001834,0.006029,0.003046,-0.000882,-0.002580,...,0.004098,-0.001065,-0.002693,0.002363,0.003390,-0.002213,-0.000915,0.001582,0.003538,-0.000345
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1994,2998.75,-0.003297,0.004280,0.005259,0.002574,-0.000131,0.004680,0.005309,-0.001902,0.003098,...,0.002951,-0.000357,0.001687,0.000543,0.002951,-0.000226,0.002559,-0.000638,0.006624,-0.002223
1995,2999.00,-0.007496,0.008841,-0.005708,0.012000,-0.005177,0.009387,-0.006992,0.009208,-0.003321,...,-0.006655,0.008747,-0.004910,0.006195,-0.008808,0.008892,-0.004979,0.006996,-0.003265,0.005413
1996,2999.25,-0.004842,0.006815,-0.011927,0.016316,-0.002276,0.006583,-0.011904,0.014013,-0.003000,...,-0.011962,0.014394,-0.004136,0.005316,-0.017188,0.015877,-0.002835,0.005830,-0.011272,0.011886
1997,2999.50,0.005060,-0.003822,-0.011849,0.014841,0.007708,-0.002991,-0.011110,0.014029,0.005001,...,-0.012392,0.015877,0.006574,-0.003925,-0.017856,0.017425,0.007721,-0.004032,-0.013584,0.014017


In [28]:
answer2.to_csv('/content/drive/My Drive/철도/answer20230826_ar_c.csv', index=False)

In [31]:
answer1 = pd.read_csv('/content/drive/My Drive/철도/answer20230826_ar_s.csv')
answer = pd.concat([answer1,answer2], axis=1)

In [32]:
answer

,Distance,YL_M1_B1_W1_s30,YR_M1_B1_W1_s30,YL_M1_B1_W2_s30,YR_M1_B1_W2_s30,YL_M1_B1_W1_s40,YR_M1_B1_W1_s40,YL_M1_B1_W2_s40,YR_M1_B1_W2_s40,YL_M1_B1_W1_s50,...,YL_M1_B1_W2_c50,YR_M1_B1_W2_c50,YL_M1_B1_W1_c70,YR_M1_B1_W1_c70,YL_M1_B1_W2_c70,YR_M1_B1_W2_c70,YL_M1_B1_W1_c100,YR_M1_B1_W1_c100,YL_M1_B1_W2_c100,YR_M1_B1_W2_c100
0,2500.25,0.028696,-0.004394,0.062964,-0.047209,0.005088,0.004656,0.000913,0.003848,0.011418,...,-0.002854,0.007505,0.010698,0.005484,-0.004044,0.008491,0.009935,0.006552,-0.003808,0.008449
1,2500.50,-0.009338,0.018618,-0.011982,0.023495,-0.001020,0.007181,0.010019,-0.005364,-0.004725,...,0.009938,-0.005174,-0.006434,0.009810,0.010753,-0.005876,-0.004632,0.008645,0.008023,-0.002847
2,2500.75,-0.004410,0.011144,0.010570,-0.000342,-0.005577,0.009786,0.003779,-0.000228,-0.007534,...,0.006340,-0.002528,-0.010609,0.010505,0.006946,-0.004182,-0.010056,0.010169,0.006999,-0.002435
3,2501.00,-0.001958,0.006689,0.004273,0.005091,-0.004674,0.008318,0.004108,-0.001283,-0.007510,...,0.005526,-0.002172,-0.007203,0.006657,0.005290,-0.003604,-0.004978,0.004826,0.004712,-0.000885
4,2501.25,0.001669,0.002719,0.005722,0.003122,-0.001619,0.004846,0.003738,-0.001576,-0.003910,...,0.004098,-0.001065,-0.002693,0.002363,0.003390,-0.002213,-0.000915,0.001582,0.003538,-0.000345
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1994,2998.75,0.002040,0.001348,0.002698,0.005167,0.002212,0.000867,0.002002,0.000246,0.001704,...,0.002951,-0.000357,0.001687,0.000543,0.002951,-0.000226,0.002559,-0.000638,0.006624,-0.002223
1995,2999.00,-0.002053,0.005068,-0.008352,0.014076,-0.002647,0.005451,-0.006235,0.007199,-0.004177,...,-0.006655,0.008747,-0.004910,0.006195,-0.008808,0.008892,-0.004979,0.006996,-0.003265,0.005413
1996,2999.25,-0.000109,0.003812,-0.010085,0.014700,-0.000649,0.003680,-0.009022,0.009268,-0.002894,...,-0.011962,0.014394,-0.004136,0.005316,-0.017188,0.015877,-0.002835,0.005830,-0.011272,0.011886
1997,2999.50,0.007347,-0.002344,-0.009122,0.013326,0.007603,-0.004494,-0.008075,0.008452,0.004040,...,-0.012392,0.015877,0.006574,-0.003925,-0.017856,0.017425,0.007721,-0.004032,-0.013584,0.014017


In [33]:
answer.to_csv('/content/drive/My Drive/철도/answer20230826_ar.csv', index=False)